# 05 - Evaluation

This notebook evaluates the fine-tuned model using BLEU, chrF++, and custom metrics.

In [ ]:
!pip install -q sacrebleu transformers

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from training.evaluation import Evaluator, load_test_set

SOURCE_LANG = "so"
TARGET_LANG = "sw"
MODEL_DIR = Path(f'../outputs/{SOURCE_LANG}-{TARGET_LANG}/final')
TEST_DIR = Path('../data/evaluation')

In [ ]:
# Load test data (Flores-200 devtest)
test_sources, test_references, _ = load_test_set(
    source_file=str(TEST_DIR / f'flores_devtest.{SOURCE_LANG}'),
    target_file=str(TEST_DIR / f'flores_devtest.{TARGET_LANG}')
)

print(f"Test set size: {len(test_sources)}")

In [ ]:
# Load fine-tuned model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR / 'merged'),
    torch_dtype=torch.bfloat16,
    device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR / 'merged'))

def translate_daraja(text):
    prompt = f"""<start_of_turn>user
Translate the following Somali text to Swahili:
{text}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('<start_of_turn>model\n')[-1].strip()

In [ ]:
# Initialize evaluator
evaluator = Evaluator(source_lang=SOURCE_LANG, target_lang=TARGET_LANG)

# Evaluate fine-tuned model
daraja_result = evaluator.evaluate_model(
    translate_daraja,
    test_sources,
    test_references,
    model_name='daraja-fine-tuned',
    test_set_name='flores200_devtest'
)

In [ ]:
# Compare with baseline (zero-shot)
def translate_baseline(text):
    # Placeholder for baseline translation
    return f"[baseline] {text}"

baseline_result = evaluator.evaluate_model(
    translate_baseline,
    test_sources,
    test_references,
    model_name='baseline-zero-shot',
    test_set_name='flores200_devtest'
)

In [ ]:
# Save results
evaluator.save_results(
    [daraja_result, baseline_result],
    str(TEST_DIR / 'evaluation_results.json')
)

evaluator.generate_report(
    [daraja_result, baseline_result],
    str(TEST_DIR / 'evaluation_report.md')
)

In [ ]:
print("=" * 50)
print("EVALUATION COMPLETE")
print("=" * 50)
print(f"\nDaraja Fine-tuned:")
print(f"  BLEU: {daraja_result.bleu:.2f}")
print(f"  chrF++: {daraja_result.chrf_plus_plus:.2f}")
print(f"\nBaseline:")
print(f"  BLEU: {baseline_result.bleu:.2f}")
print(f"  chrF++: {baseline_result.chrf_plus_plus:.2f}")
print(f"\nImprovement: +{daraja_result.bleu - baseline_result.bleu:.2f} BLEU")